In [0]:
from pyspark.sql.functions import *

df = spark.table("workspace.bronze.bronze_products")

# Cleaning the df (Selecting imp. fields only)
cleaned_df = df.select(
    col("product_id"),
    col("product_name"),
    col("category"),
    col("price"),
    col("currency"),
    col("country_code"),
    col("_modified"),
    col("ingestion_ts") 
).filter(col("product_id").isNotNull())

In [0]:
missing_values = ['\\n', '?', '', 'null']

# Adding is_valid flag
# Instead of deleting records, Adding missing values to them
cleaned_df = (
    cleaned_df
    .withColumn(
        "is_valid",
        when(
            col("product_name").isNull() |
            col("category").isNull() |
            col("country_code").isNull() |
            lower(trim(col("product_name"))).isin(missing_values) |
            lower(trim(col("country_code"))).isin(missing_values) |
            lower(trim(col("category"))).isin(missing_values),
            0
        ).otherwise(1)
    )
    .withColumn(
        "product_name",
        when(
            col("product_name").isNull() |
            lower(trim(col("product_name"))).isin(missing_values),
            None
        ).otherwise(col("product_name"))
    )
    .withColumn(
        "category",
        when(
            col("category").isNull() |
            lower(trim(col("category"))).isin(missing_values),
            None
        ).otherwise(col("category"))
    )
    .withColumn(
        "country_code",
        when(
            col("country_code").isNull() |
            lower(trim(col("country_code"))).isin(missing_values),
            None
        ).otherwise(col("country_code"))
    )
)

In [0]:
display(cleaned_df)

In [0]:
cleaned_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.silver_products")